# W01A — Introducción (DDIA Cap. 1) + Entorno local + Evidencia

## Qué haremos hoy
- Entender el “por qué” (DDIA Cap. 1): **Reliability / Scalability / Maintainability**
- Montar un entorno local reproducible (sin Docker).
- Usar SQL solo como **instrumento de evidencia** (no como tema formal aún).

## Mapa del curso (conceptual)
- **Bronze/Raw**: dato tal cual llega (trazabilidad)
- **Silver**: limpieza + tipado + reglas de calidad
- **Gold**: modelado para consulta (hechos/dimensiones/marts/métricas)

## Relación con carpetas (desde ya)
- `data/raw/`  → Bronze/Raw (archivos originales)
- `data/silver/` → outputs intermedios (Parquet/tablas exportadas)
- `data/gold/` → marts/métricas/exports listos para consumo
- `data/exoplanets.duckdb` → “warehouse local” (tablas)
- `artifacts/` → evidencia (JSON de tests, profiling, planes, logs)
- `docs/` → decisiones, runbook, glosario (mantenibilidad)


In [4]:
import os
os.getcwd()

'c:\\Users\\Ider Diaz\\Desktop\\Todos los Notebooks'

In [6]:
# 1) Setup (cross-platform)
import sys, platform, hashlib
from pathlib import Path
import duckdb
import os

os.chdir("..")

PROJECT_ROOT = Path(".").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR  = DATA_DIR / "raw"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DOCS_DIR = PROJECT_ROOT / "docs"

for d in [RAW_DIR, SILVER_DIR, GOLD_DIR, ARTIFACTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "exoplanets.duckdb"
con = duckdb.connect(str(DB_PATH))

def show(path: Path):
    return {"exists": path.exists(), "path": str(path), "size_bytes": path.stat().st_size if path.exists() else None}


print("OS:", platform.platform())
print("Python:", sys.version.split()[0])
print("DuckDB:", con.execute("SELECT version()").fetchone()[0])
show(DB_PATH)


OS: Windows-10-10.0.19045-SP0
Python: 3.11.5
DuckDB: v1.5.3


{'exists': True,
 'path': 'C:\\Users\\Ider Diaz\\data\\exoplanets.duckdb',
 'size_bytes': 12288}

## Evidencia mínima: ¿el motor responde?
Esto NO es “clase de SQL”. Es verificar que el motor funciona.

In [7]:
con.execute("SELECT 42 AS answer").fetchall()

[(42,)]

## “Experimento” controlado: una tabla pequeña
La idea: **consulta → evidencia**.

In [8]:
con.execute("CREATE OR REPLACE TABLE demo_numbers(x INTEGER)")
con.execute("INSERT INTO demo_numbers VALUES (1), (2), (3)")
con.execute("SELECT SUM(x) AS s FROM demo_numbers").fetchall()


[(6,)]

## Tu turno (en clase): 2 mini-tablas + 2 métricas
Ejecuta y pega los outputs al final del notebook.

In [9]:
# TU TURNO 1
con.execute("CREATE OR REPLACE TABLE students(name VARCHAR, semester INTEGER)")
con.execute("INSERT INTO students VALUES ('Ana', 7), ('Luis', 8), ('Sofia', 10)")
con.execute("SELECT semester, COUNT(*) n FROM students GROUP BY 1 ORDER BY 1").fetchall()


[(7, 1), (8, 1), (10, 1)]

In [17]:
# TU TURNO 2
con.execute("CREATE OR REPLACE TABLE submissions(name VARCHAR, lab VARCHAR, ok BOOLEAN)")
con.execute("""
INSERT INTO submissions VALUES
  ('Ana', 'W01', TRUE),
  ('Ana', 'W02', FALSE),
  ('Luis','W01', TRUE),
  ('Luis','W02', TRUE),
  ('Sofia','W01', TRUE)
""")
con.execute("SELECT name, COUNT(*) n FROM submissions GROUP BY 1 ORDER BY n DESC").fetchall()


[('Ana', 2), ('Luis', 2), ('Sofia', 1)]

In [11]:
try:
    con.close()
    print("DuckDB connection closed.")
except NameError:
    print("No connection named 'con' in this notebook.")


DuckDB connection closed.


# W01B — Bronze/Raw: ingesta + trazabilidad + sanity checks (DDIA Cap. 1)

## Objetivo
- Garantizar que existe un **Raw** en `data/raw/` con nombre esperado
- Registrar **hash SHA-256** (trazabilidad)
- Consultar el CSV vía **VIEW raw_ps** (sin modificar el archivo)
- Ejecutar sanity checks y guardar evidencia en `artifacts/`

> Importante: En Semana 1 **no enseñamos SQL formal**. SQL aquí es instrumento de evidencia.

In [11]:
# Setup común (cross-platform) + detección de raíz del proyecto
import sys, time, json, hashlib, platform, subprocess
from pathlib import Path
import duckdb

def find_project_root(start: Path) -> Path:
    """Busca hacia arriba una carpeta que contenga `src/` y `data/`.
    Esto evita errores cuando el notebook se ejecuta desde notebooks/."""
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / "src").exists() and (p / "data").exists():
            return p
    # fallback: carpeta actual
    return cur

PROJECT_ROOT = find_project_root(Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DOCS_DIR = PROJECT_ROOT / "docs"
for d in [RAW_DIR, ARTIFACTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "exoplanets.duckdb"
con = duckdb.connect(str(DB_PATH))

def show(path: Path):
    return {"exists": path.exists(), "path": str(path), "size_bytes": path.stat().st_size if path.exists() else None}

def sha256_file(path: Path, chunk_size: int = 1<<20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def run_module(mod: str, *args: str):
    """Ejecuta `python -m ...` desde la raíz del proyecto para que `src.*` sea importable."""
    cmd = [sys.executable, "-m", mod, *args]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(PROJECT_ROOT))

print("OS:", platform.platform())
print("Python:", sys.version.split()[0])
print("DuckDB:", con.execute("SELECT version()").fetchone()[0])


PROJECT_ROOT: C:\Users\Ider Diaz\Desktop\Todos los Notebooks
OS: Windows-10-10.0.19045-SP0
Python: 3.11.5
DuckDB: v1.5.3


## 1) Localizar el archivo Raw

Nombre esperado (por el script oficial del curso): `pscomppars.csv`

**Regla del curso:** el Raw vive en `data/raw/` y NO se edita a mano.


In [12]:
EXPECTED = "pscomppars.csv"
raw_csv = RAW_DIR / EXPECTED

# Si no está con el nombre esperado, buscamos candidatos (para no bloquear la clase)
if not raw_csv.exists():
    candidates = list(RAW_DIR.glob("pscomppars*.csv")) + list(RAW_DIR.glob("*.csv"))
    candidates = [c for c in candidates if c.is_file()]
    if candidates:
        raw_csv = sorted(candidates)[0]
        print(f"No encontré {EXPECTED}. Encontré y usaré: {raw_csv.name}")
        print("   Recomendación: renombra a pscomppars.csv para estandarizar el curso.")
    else:
        raw_csv = None

raw_csv


WindowsPath('C:/Users/Ider Diaz/Desktop/Todos los Notebooks/data/raw/pscomppars.csv')

## 2) (Opcional) Descargar el Raw con el script del curso

Si **tienes internet** y quieres descargar el Raw, ejecuta este comando en terminal **desde la raíz del proyecto**:

- Windows / mac / Linux:
`python -m src.ingest.download_exoplanets --format csv --limit 50000`

**Nota:** el notebook NO intenta descargar automáticamente (para evitar errores cuando no hay internet).


In [13]:
# Si no hay Raw, NO intentamos descargar automáticamente (para evitar fallos sin internet).
# Si quieres descargar desde clase (y tienes internet), pon DO_DOWNLOAD=True.
DO_DOWNLOAD = True

if raw_csv is None:
    if DO_DOWNLOAD:
        # Descarga oficial del curso (requiere internet). Escribe en data/raw/pscomppars.csv
        run_module("src.ingest.download_exoplanets", "--format", "csv", "--limit", "50000")
        raw_csv = RAW_DIR / "pscomppars.csv"
    else:
        raise FileNotFoundError(
            "No encontré ningún CSV en data/raw/.\n"
            "Solución A (recomendada): coloca pscomppars.csv en data/raw/.\n"
            "Solución B (con internet): cambia DO_DOWNLOAD=True y re-ejecuta esta celda."
        )

# Evidencia de trazabilidad
show(raw_csv), sha256_file(raw_csv)


({'exists': True,
  'path': 'C:\\Users\\Ider Diaz\\Desktop\\Todos los Notebooks\\data\\raw\\pscomppars.csv',
  'size_bytes': 3900657},
 'cd30dfbf742ac71ee69a7e24d6ac80f9e4126bf7e2bb9d46844637761e75a5a8')

## 3) Crear VIEW Bronze/Raw

Creamos una vista para consultar el CSV sin modificarlo.


In [14]:
path = str(raw_csv).replace("\\", "/").replace("'", "''")  # Windows-safe
con.execute(f"""
CREATE OR REPLACE VIEW raw_ps AS
SELECT * FROM read_csv_auto('{path}')
""")

In [15]:
con.execute("SELECT COUNT(*) AS n_rows FROM raw_ps").fetchall()


[(6291,)]

## 4) Sanity checks mínimos (evidencia)

Haremos 3 checks:
1) filas y columnas
2) nulos en una columna clave
3) muestra de filas


In [16]:
# Check 1
n_rows = con.execute("SELECT COUNT(*) FROM raw_ps").fetchone()[0]
n_cols = con.execute("SELECT COUNT(*) FROM pragma_table_info('raw_ps')").fetchone()[0]
n_rows, n_cols

(6291, 84)

In [17]:
# 2) nulos en una columna clave típica
con.execute("SELECT COUNT(*) AS null_pl_name FROM raw_ps WHERE pl_name IS NULL").fetchall()

[(0,)]

In [18]:
# Check 3
con.execute("SELECT pl_name, hostname, discoverymethod, disc_year FROM raw_ps WHERE pl_name IS NOT NULL LIMIT 10").fetchall()


[('11 Com b', '11 Com', 'Radial Velocity', 2007),
 ('11 UMi b', '11 UMi', 'Radial Velocity', 2009),
 ('14 And b', '14 And', 'Radial Velocity', 2008),
 ('14 Her b', '14 Her', 'Radial Velocity', 2002),
 ('16 Cyg B b', '16 Cyg B', 'Radial Velocity', 1996),
 ('17 Sco b', '17 Sco', 'Radial Velocity', 2020),
 ('18 Del b', '18 Del', 'Radial Velocity', 2008),
 ('1RXS J160929.1-210524 b', '1RXS J160929.1-210524', 'Imaging', 2008),
 ('24 Boo b', '24 Boo', 'Radial Velocity', 2018),
 ('24 Sex b', '24 Sex', 'Radial Velocity', 2010)]

## 5) Guardar evidencia en artifacts (JSON)

Esto es evaluable y reproducible.


In [19]:
artifact = {
  "raw_file": str(raw_csv),
  "raw_sha256": sha256_file(raw_csv),
  "n_rows": n_rows,
  "n_cols": n_cols
}
out = ARTIFACTS_DIR / f"w01b_raw_evidence_{int(time.time())}.json"
out.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
show(out)


{'exists': True,
 'path': 'C:\\Users\\Ider Diaz\\Desktop\\Todos los Notebooks\\artifacts\\w01b_raw_evidence_1779958055.json',
 'size_bytes': 220}

## Reflexión (DDIA Cap.1)
- ¿Qué haría que tu sistema sea “no confiable” aunque el código corra?
- ¿Qué documento te gustaría encontrar si heredas el proyecto de otro equipo?


## Entregable W01
Crea estos archivos (usa `docs/templates/` si existe):
- `docs/decisions_log.md`
- `docs/glossary.md`
- `docs/w01a_run.md`
- `docs/w01b_checks.md`: pega 3 checks (query + resultado).
- `artifacts/w01b_raw_evidence_*.json`: hash + n_rows + n_cols.

## Tarea

- Agregar 1 sanity check extra y documentarlo en docs/w01b_checks.md.

    Ejemplos permitidos:

    - duplicados por pl_name

    - valores negativos en columnas numéricas

    - años fuera de rango (ej. < 1980 o > año actual)

- Registrar una decisión en docs/decisions_log.md (tu formato exacto) sobre trazabilidad:
Ejemplo real:


```markdown
- Fecha: 2025-12-21
- Decisión: Guardar SHA-256 del CSV raw en artifacts por cada ejecución.
- Razón: Detectar cambios invisibles del dato (DDIA reliability/operability).
- Alternativas: Confiar en el nombre del archivo (rechazada), usar solo fecha de descarga (rechazada).
- Evidencia (counts, EXPLAIN, timings): raw_sha256=..., n_rows=..., n_cols=...
```


In [23]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

print("Archivo:", raw_csv)
hash_value = sha256_file(raw_csv)
print("SHA-256:", hash_value)

Archivo: C:\Users\Ider Diaz\Desktop\Todos los Notebooks\data\raw\pscomppars.csv
SHA-256: cd30dfbf742ac71ee69a7e24d6ac80f9e4126bf7e2bb9d46844637761e75a5a8


In [25]:
# Escapar comillas simples en la ruta
path = str(raw_csv).replace("\\", "/").replace("'", "''")
con.execute(f"CREATE OR REPLACE VIEW raw_ps AS SELECT * FROM read_csv_auto('{path}')")

# Obtener nombres de columnas usando PRAGMA (robusto)
col_info = con.execute("PRAGMA table_info('raw_ps')").fetchdf()
col_names = col_info['name'].tolist()
print("Columnas encontradas (primeras 5):", col_names[:5])

# Detectar columna pl_name
pl_name_col = None
for col in col_names:
    if 'pl_name' in col:
        pl_name_col = col
        break
if pl_name_col is None:
    raise KeyError("No se encontró columna con 'pl_name'")
print("Columna pl_name identificada como:", pl_name_col)

n_rows = con.execute("SELECT COUNT(*) FROM raw_ps").fetchone()[0]
print("Número total de filas:", n_rows)

Columnas encontradas (primeras 5): ['pl_name', 'hostname', 'sy_snum', 'sy_pnum', 'discoverymethod']
Columna pl_name identificada como: pl_name
Número total de filas: 6291


In [26]:
# Check 1: nulos en pl_name
null_pl = con.execute(f"SELECT COUNT(*) FROM raw_ps WHERE {pl_name_col} IS NULL").fetchone()[0]
print(f"Nulos en {pl_name_col}: {null_pl}")

# Check 2: muestra de 10 filas
sample = con.execute(f"SELECT {pl_name_col}, hostname, discoverymethod, disc_year FROM raw_ps LIMIT 10").fetchall()
print("Muestra:")
for row in sample:
    print(row)

Nulos en pl_name: 0
Muestra:
('11 Com b', '11 Com', 'Radial Velocity', 2007)
('11 UMi b', '11 UMi', 'Radial Velocity', 2009)
('14 And b', '14 And', 'Radial Velocity', 2008)
('14 Her b', '14 Her', 'Radial Velocity', 2002)
('16 Cyg B b', '16 Cyg B', 'Radial Velocity', 1996)
('17 Sco b', '17 Sco', 'Radial Velocity', 2020)
('18 Del b', '18 Del', 'Radial Velocity', 2008)
('1RXS J160929.1-210524 b', '1RXS J160929.1-210524', 'Imaging', 2008)
('24 Boo b', '24 Boo', 'Radial Velocity', 2018)
('24 Sex b', '24 Sex', 'Radial Velocity', 2010)


In [29]:
duplicates = con.execute(f"""
    SELECT {pl_name_col}, COUNT(*) AS c
    FROM raw_ps
    WHERE {pl_name_col} IS NOT NULL
    GROUP BY {pl_name_col}
    HAVING COUNT(*) > 1
""").fetchall()
n_duplicates = len(duplicates)
print(f"Planetas duplicados (por {pl_name_col}): {n_duplicates}")
if n_duplicates == 0:
    print("No hay duplicados. La columna es candidata a clave primaria.")
else:
    print("Advertencia: existen duplicados. Ejemplo:", duplicates[:3])

Planetas duplicados (por pl_name): 0
No hay duplicados. La columna es candidata a clave primaria.


In [30]:
artifact = {
    "raw_file": str(raw_csv),
    "raw_sha256": hash_value,
    "n_rows": n_rows,
    "n_cols": len(col_names),
    "null_pl_name": null_pl,
    "duplicate_pl_name_count": n_duplicates
}
out = ARTIFACTS_DIR / f"w01b_raw_evidence_{int(time.time())}.json"
out.write_text(json.dumps(artifact, indent=2))
print("Evidencia guardada en:", out)

Evidencia guardada en: C:\Users\Ider Diaz\Desktop\Todos los Notebooks\artifacts\w01b_raw_evidence_1779959305.json


In [35]:
checks_md = f"""# W01B - Sanity checks

## Check 1: Filas y columnas
- Filas: {n_rows}
- Columnas: {len(col_names)}

## Check 2: Nulos en {pl_name_col}
- Resultado: {null_pl} nulos

## Check 3: Muestra de 10 planetas
- (Ver salida del notebook)

## Check extra (tarea): Duplicados por {pl_name_col}
- Número de planetas duplicados: {n_duplicates}
- Conclusión: {'No hay duplicados, la columna es única.' if n_duplicates == 0 else 'Existen duplicados, revisar fuente de datos.'}
"""
checks_path = DOCS_DIR / "w01b_checks.md"
checks_path.write_text(checks_md, encoding="utf-8")
print("Documentación de checks guardada en:", checks_path)

# %% [markdown]
# ## 7) Registrar decisión en docs/decisions_log.md
from datetime import datetime
# %% [code]
decision_entry = f"""- Fecha: {datetime.now().strftime('%Y-%m-%d')}
- Decisión: Guardar SHA-256 del CSV raw en artifacts por cada ejecución.
- Razón: Detectar cambios invisibles del dato (DDIA reliability/operability). Además, se añadió un check de duplicados para garantizar la unicidad de pl_name.
- Alternativas: Confiar en el nombre del archivo (rechazada), usar solo fecha de descarga (rechazada).
- Evidencia: raw_sha256={hash_value}, n_rows={n_rows}, n_cols={len(col_names)}, duplicate_count={n_duplicates}
"""
log_path = DOCS_DIR / "decisions_log.md"
# Si el archivo no existe, crearlo con cabecera
if not log_path.exists():
    log_path.write_text("# Decisions Log\n\n", encoding="utf-8")
with open(log_path, "a", encoding="utf-8") as f:
    f.write(decision_entry + "\n")
print("Decisión registrada en:", log_path)

# %% [markdown]
# ## 8) Cierre

# %% [code]
con.close()
print("DuckDB connection closed.")

Documentación de checks guardada en: C:\Users\Ider Diaz\Desktop\Todos los Notebooks\docs\w01b_checks.md
Decisión registrada en: C:\Users\Ider Diaz\Desktop\Todos los Notebooks\docs\decisions_log.md
DuckDB connection closed.
